In [1]:
import pandas as pd

# Load the datasets
try:
    train_df = pd.read_csv(r'/data/womens_uk/3.with_risk\risk_train_data.csv', encoding='latin1', on_bad_lines='skip')
    test_df = pd.read_csv(r'/data/womens_uk/3.with_risk\risk_test_data.csv', encoding='latin1', on_bad_lines='skip')

    print("Predicted risk training data loaded successfully.")
    print("Predicted risk data loaded successfully.")

except FileNotFoundError as e:
    print(e)
    print("\n Please make sure the files 'risk_training_data.csv' and 'risk_test_data.csv' are uploaded.")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8456\3456825233.py:7: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv(r'C:\Users\ADMIN\return_risk\data\womens_uk\with_risk\risk_test_data.csv', encoding='latin1', on_bad_lines='skip')


Predicted risk training data loaded successfully.
Predicted risk data loaded successfully.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the TF-IDF vectorizer
vectorizer_care = TfidfVectorizer(stop_words='english', max_features=100)
vectorizer_desc = TfidfVectorizer(stop_words='english', max_features=100)
vectorizer_name = TfidfVectorizer(stop_words='english', max_features=100)

# Fit the vectorizers to the training data and transform it
X_train_care = vectorizer_care.fit_transform(train_df['Care information'])
X_train_desc = vectorizer_desc.fit_transform(train_df['Description'])
X_train_name = vectorizer_name.fit_transform(train_df['Name'])


# Transform the test data
X_test_care = vectorizer_care.transform(test_df['Care information'])
X_test_desc = vectorizer_desc.transform(test_df['Description'])
X_test_name = vectorizer_name.transform(test_df['Name'])

# Convert the transformed data to a pandas DataFrame
X_train_care = pd.DataFrame(X_train_care.toarray(), columns=vectorizer_care.get_feature_names_out())
X_train_desc = pd.DataFrame(X_train_desc.toarray(), columns=vectorizer_desc.get_feature_names_out())
X_train_name = pd.DataFrame(X_train_name.toarray(), columns=vectorizer_name.get_feature_names_out())

X_test_care = pd.DataFrame(X_test_care.toarray(), columns=vectorizer_care.get_feature_names_out())
X_test_desc = pd.DataFrame(X_test_desc.toarray(), columns=vectorizer_desc.get_feature_names_out())
X_test_name = pd.DataFrame(X_test_name.toarray(), columns=vectorizer_name.get_feature_names_out())


# Concatenate the transformed data with the original data
train_df = pd.concat([train_df, X_train_care, X_train_desc, X_train_name], axis=1)
test_df = pd.concat([test_df, X_test_care, X_test_desc, X_test_name], axis=1)

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Define the features to use for training the model
features = ['Segment', 'Category', 'Color', 'Full Price ($)', 'Current Discount Percentage'] + list(X_train_care.columns) + list(X_train_desc.columns) + list(X_train_name.columns)

# Create the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Segment', 'Category', 'Color']),
        ('num', StandardScaler(), ['Full Price ($)', 'Current Discount Percentage'])
    ],
    remainder='passthrough'
)

# Create the model pipeline
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, solver='liblinear'))
])

# Split the data into training and testing sets
X_train = train_df[features]
y_train = train_df['return_risk']
X_test = test_df[features]
y_test = test_df['return_risk']

# Train the model
model.fit(X_train, y_train)

# Make predictions on the test data
y_pred = model.predict(X_test)

# Evaluate the model's performance
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

C:\Users\ADMIN\return_risk\gtv\Lib\site-packages\sklearn\linear_model\_logistic.py:1288: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Accuracy: 0.915794391158995

Classification Report:
              precision    recall  f1-score   support

        High       0.90      0.90      0.90     29936
         Low       0.90      0.94      0.92     22641
      Medium       0.96      0.91      0.93     15922

    accuracy                           0.92     68499
   macro avg       0.92      0.92      0.92     68499
weighted avg       0.92      0.92      0.92     68499


Confusion Matrix:
[[27064  2310   562]
 [ 1440 21185    16]
 [ 1433     7 14482]]


In [7]:
import joblib
import os
# Save the model to a file
joblib.dump(model, 'logistic_regression_model.joblib')
# Create the models directory if it doesn't exist
if not os.path.exists('../../app/models'):
    os.makedirs(r'../../app/models')

# Save the model to the models directory
joblib.dump(model, '../../models/logistic_regression_model.joblib')

['models/logistic_regression_model.joblib']

In [10]:
# Define the directory
output_directory = r'C:\Users\ADMIN\return_risk\data\womens_uk\4.prediction\model1'

# Ensure the directory exists
os.makedirs(output_directory, exist_ok=True)

# Define the full file paths, including filenames
output_train_filepath = os.path.join(output_directory, 'actual_risk_train_data.csv')
output_test_filepath = os.path.join(output_directory, 'actual_risk_test_data.csv')

# Save the cleaned DataFrames to CSV files
train_df.to_csv(output_train_filepath, index=False)
test_df.to_csv(output_test_filepath, index=False)

print(f"Actual risk training data saved to: {output_train_filepath}")
print(f"Actual risk test data saved to: {output_test_filepath}")

Actual risk training data saved to: C:\Users\ADMIN\return_risk\data\womens_uk\4.prediction\model1\actual_risk_train_data.csv
Actual risk test data saved to: C:\Users\ADMIN\return_risk\data\womens_uk\4.prediction\model1\actual_risk_test_data.csv
